# D1 · Height on the MVP ballot, against the season's top 50 scorers

## The question

> Produce the height distribution of players on the Michael Jordan Trophy list
> compared with the top 50 players of the season, using season stats from
> 2019-20 through the end of 2023-24. (No need to examine each season
> separately.)

## Reading the question

Three phrases have to be pinned down before anything can be queried.

**"The Michael Jordan Trophy list."** The trophy is the NBA's Most Valuable
Player award, renamed after Jordan in 2022. Only one player wins it each year,
so a list of winners across five seasons would be five names, not a
distribution. The MVP *ballot* is the list that fits: roughly a hundred voters
each rank five players, and everyone who collects at least one vote appears on
the published result. That comes to between 9 and 15 names a season.

**"The top 50 players of the season."** This database has no wins column and no
all-round rating, so "top 50" cannot mean the 50 best. It means the 50 highest
point scorers, which is the order Basketball-Reference sorts its own season
pages in. Everything below is therefore about scoring volume. A defensive
specialist can be one of the ten most valuable players in the league and never
come near this list.

**"2019-20 through the end of 2023-24."** Seasons are stored under their ending
year, so the window is 2020 to 2024 inclusive. The brief says the seasons need
not be examined separately, so all five are pooled.

Pooling has a cost worth naming. A player who made the ballot in four of the
five seasons contributes four rows, and Nikola Jokić is not four different
people. Height is the one attribute that never changes from season to season,
which makes the cost easy to measure: collapse to one row per player and run the
comparison again. Both versions are below, and they disagree. That turns out to
be the most interesting thing in the notebook.

**The two groups overlap, and the overlap stays.** Most MVP candidates are also
top-50 scorers. The question names two populations and asks for a comparison. It
does not ask for the league to be cut in half. Stripping the overlap out would
leave a handful of rows in the ballot group and answer a question nobody asked.
So a player-season can sit in both groups, and the size of the overlap is
reported below rather than engineered away.

**What is probably doing the work.** Height in basketball is largely position.
Point guards average about 188 cm and centres about 211 cm, a 23 cm gap inside
the same league. If one group holds more guards, or more centres, than the
other, its mean height moves for that reason alone and nothing has been learned
about MVP voting. Position is reported alongside height throughout for exactly
that reason.

## What we need, and where it comes from

Two lists of players, for each of the five seasons from 2019-20 to 2023-24.

The first list is everyone who received at least one MVP vote that season. The
second is the fifty players who scored the most points that season. For each of
them we need three things: how tall he is, which position he played that season,
and which of the two lists he belongs to. He can be on both.

All of it is already in one place. The database keeps a single table with one
row per player per season, and that row carries the season's scoring rank and
the MVP-vote marker next to the player's height and position. Nothing needs
joining, and nothing needs converting: heights were turned into centimetres when
the data was first cleaned.

In [1]:
import _setup  # noqa: F401

import pandas as pd

from utils.custom_plots import cross_tab_heatmap, ecdf_plot, grouped_box_plot
from utils.custom_stats import compare_groups, summary_stats
from utils.db_utils import run_query

In [2]:
SQL = """
SELECT season,
       season_label,
       player_id,
       player_name,
       position,
       height_cm,
       points_rank,
       is_mvp_candidate,
       mvp_rank
FROM analyst_ready.player_season
WHERE season BETWEEN 2020 AND 2024
  AND (is_mvp_candidate OR points_rank <= 50)
ORDER BY season, points_rank
"""

players = run_query(SQL)

# SQL `numeric` arrives as Decimal on some drivers, and the plotting helpers
# skip object columns silently. Cast once, here, and the problem never appears.
players["height_cm"] = players["height_cm"].astype(float)

print(players.shape)
players.head()

(254, 9)


,season,season_label,player_id,player_name,position,height_cm,points_rank,is_mvp_candidate,mvp_rank
0,2020,2019-20,hardeja01,James Harden,SG,195.6,1,True,3.0
1,2020,2019-20,lillada01,Damian Lillard,PG,188.0,2,True,8.0
2,2020,2019-20,bookede01,Devin Booker,SG,195.6,3,False,NaN
3,2020,2019-20,antetgi01,Giannis Antetokounmpo,PF,210.8,4,True,1.0
4,2020,2019-20,youngtr01,Trae Young,PG,188.0,5,False,NaN


In [3]:
# Order positions shortest to tallest so every chart below reads left to right.
POSITIONS = ["PG", "SG", "SF", "PF", "C"]
players["position"] = pd.Categorical(
    players["position"], categories=POSITIONS, ordered=True
)

on_ballot = players["is_mvp_candidate"]
in_top50 = players["points_rank"] <= 50

# One row per group a player-season belongs to. A player-season on both lists
# appears twice, once under each label. That is the overlap, kept on purpose.
groups = pd.concat(
    [
        players.loc[on_ballot].assign(group="MVP ballot"),
        players.loc[in_top50].assign(group="Top-50 scorers"),
    ],
    ignore_index=True,
)

composition = pd.DataFrame(
    {
        "player_seasons": [
            int(on_ballot.sum()),
            int(in_top50.sum()),
            int((on_ballot & in_top50).sum()),
            int((on_ballot & ~in_top50).sum()),
        ],
        "distinct_players": [
            players.loc[on_ballot, "player_id"].nunique(),
            players.loc[in_top50, "player_id"].nunique(),
            players.loc[on_ballot & in_top50, "player_id"].nunique(),
            players.loc[on_ballot & ~in_top50, "player_id"].nunique(),
        ],
    },
    index=["MVP ballot", "Top-50 scorers", "on both lists", "ballot only"],
)
print(composition, end="\n\n")

print("On the ballot without a top-50 scoring season:")
print(players.loc[on_ballot & ~in_top50,
                  ["season_label", "player_name", "position", "height_cm",
                   "points_rank", "mvp_rank"]].to_string(index=False), end="\n\n")

players.groupby("season_label", observed=True).agg(
    ballot=("is_mvp_candidate", "sum"),
    top_50=("points_rank", lambda r: int((r <= 50).sum())),
)

                player_seasons  distinct_players
MVP ballot                  61                29
Top-50 scorers             250               100
on both lists               57                26
ballot only                  4                 4

On the ballot without a top-50 scoring season:
season_label  player_name position  height_cm  points_rank  mvp_rank
     2020-21  Rudy Gobert        C      215.9           57      10.0
     2020-21  Ben Simmons       PG      208.3           87      12.0
     2020-21 Derrick Rose       PG      190.5          120       9.0
     2021-22   Chris Paul       PG      182.9           83       9.0



,ballot,top_50
season_label,,
2019-20,12,50
2020-21,15,50
2021-22,12,50
2022-23,13,50
2023-24,9,50


The MVP ballot holds 61 player-seasons across the five years, produced by 29
different players. The top-50 group holds 250 by construction, 50 a season
exactly, produced by 100 different players.

57 of the 61 ballot appearances are also top-50 scoring seasons. Only four are
not, and all four sit in 2020-21 and 2021-22: Rudy Gobert, Ben Simmons, Derrick
Rose and Chris Paul. Gobert is the clearest illustration of what the scoring
definition misses. He finished 57th in points that season and 10th in the MVP
vote, because he was the best defensive player in the league.

So the "two groups" are really one group plus a superset of it, with four
exceptions. That is worth holding on to before reading any difference between
them as a difference between distinct populations.

The ballot is also small and it moves: 12, 15, 12, 13, then 9 names. The 2023-24
ballot is the shortest in the window, which is a voting-panel artefact rather
than anything about the league.

## The two distributions

An empirical cumulative distribution is the plainest way to put two height
distributions side by side. Each curve climbs from 0 to 1 as you move right, and
its height at any point reads as "this share of the group is at most this tall".

It is also the right chart for this particular variable. Basketball-Reference
publishes height in whole inches, so `height_cm` is a converted figure that only
ever lands on one of 16 rungs, 2.54 cm apart. A histogram with bins narrower
than that shows spikes separated by empty gaps, which describes the ruler rather
than the players. An ECDF needs no bin width and sidesteps the problem.

In [4]:
ecdf_plot(
    groups,
    cols="height_cm",
    group_col="group",
    mark_percentiles=[0.25, 0.5, 0.75],
    title="Height: MVP ballot vs top-50 scorers, 2019-20 to 2023-24",
)

The curves are not a shifted copy of each other. Below about 193 cm they sit on
top of one another. Both groups bottom out at the same 182.9 cm, and both reach
their lower quartile at exactly 193.0 cm. The MVP ballot is not short of short
players.

Above that point the red curve falls away from the blue one and never comes
back. The ballot's median is 203.2 cm against 198.1 cm for the top 50, and its
upper quartile is 210.8 cm against 203.2 cm. A quarter of the ballot is taller
than three quarters of the top-50 group.

The difference therefore lives entirely in the upper half of the distribution.
That is already a hint that this is about which positions attract MVP votes
rather than about height as such.

In [5]:
height_summary = pd.concat(
    [
        summary_stats(sub, cols=["height_cm"]).assign(group=name)
        for name, sub in groups.groupby("group", observed=True)
    ],
    ignore_index=True,
)

height_summary[
    ["group", "n", "mean", "ci_low", "ci_high", "median", "std", "iqr",
     "skew", "min", "max"]
].round(2)

,group,n,mean,ci_low,ci_high,median,std,iqr,skew,min,max
0,MVP ballot,61,200.70,198.29,203.12,203.2,9.44,17.8,-0.34,182.9,215.9
1,Top-50 scorers,250,198.35,197.31,199.39,198.1,8.35,10.2,0.26,182.9,223.5


The mean gap is 2.35 cm: 200.70 cm on the ballot against 198.35 cm in the top
50. Under an inch. The median gap is larger at 5.1 cm, which is the upper-half
effect the curves already showed.

Worth saying plainly: 2.35 cm is smaller than one rung of the measurement scale.
Heights come from the source in whole inches, so 2.54 cm is the finest
distinction any single player's height can express. The difference between two
group means can legitimately fall below that, since a mean of 61 discrete values
is not itself discrete, but it is not a difference anyone could see on a court.

The spread column is the more interesting one. The ballot's interquartile range
is 17.8 cm against 10.2 cm for the top 50. The ballot is not a taller group so
much as a wider one, reaching further in both directions from a similar floor.
The skew signs agree: the ballot leans negative, a long tail of shorter players
below a tall bulk, while the top 50 leans positive.

The two confidence intervals on the mean overlap (198.3 to 203.1 against 197.3
to 199.4), which is the first sign that 2.35 cm may not survive a formal test.

In [6]:
height_test = compare_groups(
    groups, group_col="group", value_col="height_cm",
    test="auto", ci="bootstrap",
)
height_test.T

,0
comparison,MVP ballot vs Top-50 scorers
n_groups,2
group_sizes,MVP ballot=61; Top-50 scorers=250
test,mannwhitney
n,311
statistic,8888.0
p_value,0.043931
decision,reject H₀
estimate_type,median difference
estimate,5.1


`chosen_because` says `auto: normal 0/2, variances equal`, so neither group
passed a normality screen and the routine went to Mann-Whitney.

That is a reasonable landing place, but not for the reason a reader might
assume. Height here takes 16 distinct values on a 2.54 cm grid, and a
Shapiro-Wilk test rejects a discrete variable almost on sight, whatever its
shape. The mixture of five position distributions underneath is a second reason.
Neither of them is skew, so the rejection should not be read as "the data are
badly shaped".

Mann-Whitney also carries a condition this data does not meet.
**It reads as a median difference only when the two distributions have the same
shape.** These two do not: the interquartile ranges are 17.8 cm and 10.2 cm, and
the skews point opposite ways. So the honest reading of this row is the weaker
claim the test always supports, which is that a player drawn at random from the
ballot tends to be taller than one drawn from the top 50. The 5.1 cm "median
difference" describes the two medians. It does not estimate a shift.

The numbers, read plainly:

- p = 0.044. Below 0.05, and not by much. Group A is the MVP ballot, so the
  positive estimate points the same way the chart does.
- Cliff's delta = 0.166, labelled small. In words: pick one player-season from
  each group at random and the ballot player is taller 58 % of the time. A coin
  flip is 50 %.
- The bootstrap interval on the median difference runs from 0.0 to 7.6 cm and
  touches zero. Nothing about "significant at 0.05" should be read as a firm
  number here.

`flags` came back empty, so the routine found nothing it wanted to warn about.

### What pooling five seasons is doing to that p-value

61 ballot rows come from 29 players. The test above treats them as 61
independent observations, which they are not. Height never changes, so a player
who makes the ballot five times contributes the same number five times and the
group looks five times as certain about him as it should.

Because height is fixed per player, the check is exact rather than approximate:
collapse each group to one row per player and run the same comparison.

In [7]:
repeat_ballot = (
    players.loc[on_ballot]
    .groupby(["player_name", "height_cm"], observed=True)
    .size()
    .rename("ballot_seasons")
    .reset_index()
    .sort_values(["ballot_seasons", "height_cm"], ascending=[False, False])
)
print(repeat_ballot.head(8).to_string(index=False), end="\n\n")

TALL_CM = 208.0
for label, mask in (("MVP ballot", on_ballot), ("Top-50 scorers", in_top50)):
    tall = players.loc[mask & (players["height_cm"] >= TALL_CM)]
    print(f"{label:<15} {len(tall):>3} of {int(mask.sum()):>3} rows "
          f"({len(tall) / mask.sum():.0%}) come from players >= {TALL_CM:.0f} cm, "
          f"{tall['player_id'].nunique()} players")
print()

per_player = groups.drop_duplicates(subset=["group", "player_id"])
compare_groups(per_player, group_col="group", value_col="height_cm",
               test="auto").T

          player_name  height_cm  ballot_seasons
Giannis Antetokounmpo      210.8               5
         Nikola Jokić      210.8               5
          Luka Dončić      203.2               5
         Jayson Tatum      203.2               4
          Joel Embiid      213.4               3
         LeBron James      205.7               3
        Stephen Curry      188.0               3
           Chris Paul      182.9               3

MVP ballot       20 of  61 rows (33%) come from players >= 208 cm, 8 players
Top-50 scorers   43 of 250 rows (17%) come from players >= 208 cm, 17 players



,0
comparison,MVP ballot vs Top-50 scorers
n_groups,2
group_sizes,MVP ballot=29; Top-50 scorers=100
test,mannwhitney
n,129
statistic,1580.0
p_value,0.463115
decision,fail to reject H₀
estimate_type,median difference
estimate,0.0


The gap does not survive.

One row per player: 199.35 cm on the ballot against 198.00 cm in the top 50, a
mean gap of 1.35 cm. The medians are identical at 198.1 cm. p = 0.46, Cliff's
delta 0.09, labelled negligible. Fail to reject.

The repeat table shows why. Giannis Antetokounmpo (210.8 cm) and Nikola Jokić
(210.8 cm) are on all five ballots. Luka Dončić is too, Jayson Tatum on four,
Joel Embiid (213.4 cm) on three. Twenty of the 61 ballot rows, a third of the
group, belong to eight players of 208 cm or more. The equivalent share of the
top-50 group is 43 rows out of 250, or 17 %.

So the pooled p = 0.044 is not measuring a preference among MVP voters for tall
players. It is measuring the fact that a handful of very tall players were the
best in the league five years running, and each of them was counted five times.
Both statements may be true. Only the second is what the test saw.

## Is any of this about height, or is it all about position?

Height in the NBA is close to a restatement of position. If the two groups are
built from different position mixes, their heights differ automatically and the
comparison above says nothing that "the ballot has more centres" would not have
said more directly.

Two things settle it. First, what each group is made of. Second, whether the
ballot is taller than the top 50 *within* a position, where the mix cannot
matter.

In [8]:
cross_tab_heatmap(
    groups, col1="group", col2="position",
    normalize="row", show_counts=True,
    title="Position mix of each group, 2019-20 to 2023-24",
    height=420,
)

The mixes are not remotely the same, and the ballot's is the strange one.

| | PG | SG | SF | PF | C |
| --- | ---: | ---: | ---: | ---: | ---: |
| MVP ballot | 42.6 % | 6.6 % | 8.2 % | 23.0 % | 19.7 % |
| Top-50 scorers | 26.8 % | 24.0 % | 16.8 % | 19.6 % | 12.8 % |

The ballot loads up at both ends of the height range and empties out in the
middle. Point guards, the shortest position, take 42.6 % of it against 26.8 % of
the top 50. Centres, the tallest, take 19.7 % against 12.8 %. The two wing
positions in between, shooting guard and small forward, hold 40.8 % of the top
50 and only 14.8 % of the ballot.

That is the wide, flat-middled distribution from the first chart, explained. MVP
votes concentrate on players who either run the offence or dominate near the
hoop, and those are the two ends of the height scale. Scoring volume is spread
across all five positions, because scoring is what every position does.

It also means the ballot is being pulled taller and shorter at once. The net
2.35 cm is what is left once those two pulls partly cancel.

In [9]:
groups["position_group"] = groups["position"].astype(str) + " · " + groups["group"]

grouped_box_plot(
    groups, group_col="position_group", value_col="height_cm",
    sort_by="median", orientation="horizontal", min_n_flag=30,
    title="Height within each position, MVP ballot vs top-50 scorers",
    height=680,
)

In [10]:
by_position = pd.concat(
    [
        summary_stats(sub, cols=["height_cm"]).assign(position=pos, group=grp)
        for (pos, grp), sub in groups.groupby(["position", "group"], observed=True)
    ],
    ignore_index=True,
)
table = (by_position
         .pivot(index="position", columns="group", values=["n", "mean"])
         .reindex(POSITIONS))
table[("gap_cm", "")] = (table[("mean", "MVP ballot")]
                         - table[("mean", "Top-50 scorers")])
print(table.round(1).to_string(), end="\n\n")

# Re-weight the ballot's per-position means onto the top-50 group's position mix.
# The two means then differ only in height-within-position, not in composition.
mix = table[("n", "Top-50 scorers")] / table[("n", "Top-50 scorers")].sum()
matched = float((table[("mean", "MVP ballot")] * mix).sum())
means = groups.groupby("group", observed=True)["height_cm"].mean()
print(f"observed gap                         {means['MVP ballot'] - means['Top-50 scorers']:5.2f} cm")
print(f"gap after matching the position mix  {matched - means['Top-50 scorers']:5.2f} cm")

                  n                      mean                gap_cm
group    MVP ballot Top-50 scorers MVP ballot Top-50 scorers       
position                                                           
PG             26.0           67.0      193.9          191.2    2.7
SG              4.0           60.0      193.0          193.2   -0.1
SF              5.0           42.0      200.1          200.2   -0.1
PF             14.0           49.0      206.8          205.1    1.7
C              12.0           32.0      211.0          210.3    0.8

observed gap                          2.35 cm
gap after matching the position mix   1.12 cm


Within a position the two groups are nearly the same height. The gaps run
+0.8 cm at centre, +1.7 cm at power forward, +2.7 cm at point guard, and
-0.1 cm at both wing positions. Every one of them is smaller than the 2.35 cm
headline, four of the five are inside one rung of the inch grid, and two point
the other way.

Holding the position mix constant makes that arithmetic explicit. Re-weight the
ballot's per-position means onto the top-50 group's position shares and the gap
falls from 2.35 cm to 1.12 cm. **Roughly half of the headline difference is
composition rather than height.** The rest is the mild within-position tilt,
most of it at point guard, where the ballot averages 193.9 cm against 191.2 cm.
Luka Dončić (203.2 cm), LeBron James (205.7 cm) and Ben Simmons (208.3 cm) each
appear on a ballot listed at point guard in this window, and nobody that size is
a typical one.

Two things in the chart need explaining before they mislead anyone.

Every ballot box carries a gold outline, which is the plotting helper flagging n
below 30. Shooting guard has four ballot rows and small forward five. Those two
cells are illustration, not evidence, and no per-position test is worth running
on them.

One box shows outlier dots: two top-50 point guards at 205.7 cm, LeBron James
and Ben Simmons. They are flagged because that group's interquartile range is
6.3 cm, about two and a half rungs of the inch grid, which puts the Tukey fence
barely one rung above the box. On a variable this coarse the fence catches the
next step up rather than anything genuinely extreme.

## Conclusion

**The MVP ballot is a slightly taller group than the season's top 50 scorers,
and a much wider one. Neither difference is large, and the taller part mostly
disappears once you stop counting the same players five times.**

| | MVP ballot | Top-50 scorers |
| --- | ---: | ---: |
| Player-seasons | 61 | 250 |
| Distinct players | 29 | 100 |
| Mean height | 200.70 cm | 198.35 cm |
| Median height | 203.2 cm | 198.1 cm |
| Interquartile range | 17.8 cm | 10.2 cm |
| Shortest / tallest | 182.9 / 215.9 cm | 182.9 / 223.5 cm |

Pooled over player-seasons, Mann-Whitney gives p = 0.044 with a small effect
(Cliff's delta 0.166) and a bootstrap interval on the median difference that
touches zero. Collapse to one row per player and the same test gives p = 0.46,
a negligible effect, and identical medians. The first result is real arithmetic
on a group where Antetokounmpo, Jokić, Dončić, Tatum and Embiid supply a third
of the rows. It is not evidence about players in general.

The spread difference is the sturdier finding, and the position mix says why.
Point guards take 42.6 % of the ballot and centres 19.7 %, against 26.8 % and
12.8 % of the top 50, while the two wing positions collapse from 40.8 % to
14.8 %. MVP votes go to the players who run an offence or own the area near the
hoop, and those are the shortest and tallest jobs on the floor. Scoring volume
is spread evenly across all five positions, because scoring is what all five do.

Within a position the two groups are close: gaps of +0.8, +1.7, +2.7, -0.1 and
-0.1 cm. Matching the ballot to the top 50's position mix cuts the 2.35 cm
headline to 1.12 cm. So the answer to "are MVP candidates taller?" is: barely,
and mostly because more of them play the two positions at the ends of the height
range.

Limits worth stating:

- "Top 50" means the 50 highest point scorers, the only ranking this data
  supports. It is not the 50 best players. Rudy Gobert finished 57th in scoring
  and 10th in the MVP vote in 2020-21, which is the whole gap in one line.
- Heights are published in whole inches, so every figure here rests on a 2.54 cm
  grid. The 2.35 cm headline gap and the 1.12 cm position-matched gap are both
  below one rung of it.
- The two groups overlap almost completely: 57 of 61 ballot rows are also
  top-50 rows. Any difference between them comes from a four-row remainder plus
  the different weight the same players carry in each group.
- Five seasons, 29 ballot players, two of them on every single ballot. This
  window is too narrow to say anything general about how MVP voting behaves.
- Nothing here is causal. Height, position and MVP votes move together, and
  which way the arrows point is not a question this data can answer.